# Load Libraries 

In [1]:
import pandas as pd  # Library for handling data (DataFrames)
import numpy as np   # Library for numerical operations
import faiss         # Library for fast similarity search (vector search)
import pickle        # Used to save Python objects (serialization)
import os            # Used for file and directory operations
from sentence_transformers import SentenceTransformer, CrossEncoder  # Models for embeddings and re-ranking
from rank_bm25 import BM25Okapi  # BM25 algorithm for keyword-based search

# Load Data

In [2]:
df = pd.read_excel(r"D:\Fall 2026 (4)\CS 1st Semester\CS492 Project\Data Set\Plant Recommendations.xlsx")  # Load dataset from Excel file

df["clean_name"] = df["Disease"].str.replace("___", " ")  # Clean disease names by replacing underscores with spaces

# Combine multiple columns into one text field to improve search quality
df["combined_text"] = (
    df["clean_name"] + " " + df["clean_name"] + " " + df["clean_name"] + " " +  # Repeat disease name to give it higher importance
    df["Disease Definition"].fillna("") + " " +  # Add disease definition (fill missing with empty)
    df["Treatment"].fillna("") + " " +           # Add treatment info
    df["Dosage"].fillna("") + " " +              # Add dosage info
    df["Treatment Steps"].fillna("")             # Add treatment steps
)

texts = df["combined_text"].tolist()  # Convert all combined text into a list


# Plant Detection


In [3]:
plants = [
    "apple", "tomato", "grape", "bell pepper", "potato", "blueberry" , "strawberry" , "sugercane" , "Watermelon" ,
    "corn", "cassava", "coffee", "rice", "cherry" , "rose" , "Orange" , "raspberry" , "squash" , "soybean" , "peach"
]  # List of supported plant names

def detect_plant(query):
    q = query.lower()  # Convert user query to lowercase for case-insensitive matching
    
    for p in plants:  # Loop through each plant in the list
        if p in q:  # Check if plant name exists in the query
            return p  # Return the detected plant name
    
    return None  # Return None if no plant is found in the query


# Detect Disease Type


In [4]:
def detect_type(text):
    text = str(text).lower()  # Convert text to lowercase for matching

    if "virus" in text:
        return "virus"  # If text contains "virus"
    elif "bacteria" in text or "bacterial" in text:
        return "bacterial"  # If text contains bacterial keywords
    elif "mite" in text or "pest" in text or "insect" in text:
        return "pest"  # If text indicates insects or pests
    else:
        return "fungal"  # Default type

df["type"] = df["Disease Definition"].apply(detect_type)  # Apply function to classify each disease


# BM25


In [5]:
tokenized_corpus = [text.lower().split() for text in texts]  # Tokenize each text into words
bm25 = BM25Okapi(tokenized_corpus)  # Initialize BM25 model


# Embedding Model


In [6]:
embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")  # Load embedding model

embeddings = embedding_model.encode(texts, normalize_embeddings=True)  # Convert texts into vectors

dimension = embeddings.shape[1]  # Get vector dimension
index = faiss.IndexFlatIP(dimension)  # Create FAISS index using inner product (cosine similarity)
index.add(np.array(embeddings))  # Add embeddings to FAISS index


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Re-ranker


In [7]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # Load cross-encoder model for re-ranking


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Hybrid Search


In [8]:
def hybrid_search(query, top_k=5):
    tokenized_query = query.lower().split()  # Convert query to lowercase and split into words (tokens)

    bm25_scores = bm25.get_scores(tokenized_query)  # Get BM25 relevance scores for all documents based on the query

    q_emb = embedding_model.encode([query], normalize_embeddings=True)  # Convert query into embedding vector (for semantic search)

    _, faiss_indices = index.search(q_emb, top_k)  # Search FAISS index and get top_k most similar results

    faiss_indices = faiss_indices[0]  # Extract indices from FAISS result (first query only)

    candidate_indices = list(set(
        faiss_indices.tolist() +  # Convert FAISS indices to list

        np.argsort(bm25_scores)[-top_k:].tolist()  # Get indices of top_k highest BM25 scores
    ))
    # Merge BM25 and FAISS results and remove duplicates

    results = df.iloc[candidate_indices].copy()  # Retrieve candidate rows from DataFrame

# Plant Filtering
    plant = detect_plant(query)  # Detect plant name from user query

    if plant:  # If a plant is detected

        filtered = results[results["clean_name"].str.lower().str.contains(plant)]  # Filter results to keep only rows that contain the detected plant

        if len(filtered) > 0:  # If filtered results are not empty
            results = filtered  # Replace results with filtered ones

    return results  # Return final results after hybrid search and filtering

# Re-ranking (FULL)


In [9]:
def rerank_full(query, results):
    pairs = [[query, row["combined_text"]] for _, row in results.iterrows()]  # Create (query, text) pairs
    scores = reranker.predict(pairs)  # Predict relevance scores

    results["score"] = scores  # Add scores to DataFrame
    results = results.sort_values(by="score", ascending=False)  # Sort by best score

    return results  # Return sorted results


# Retrieve Top Results


In [10]:
def retrieve_with_confidence(query, top_k=5, threshold=0.15):
    results = hybrid_search(query, top_k=top_k)  # Get initial candidates using hybrid search (BM25 + FAISS)

    results = rerank_full(query, results)  # Re-rank results using CrossEncoder for better accuracy

    if len(results) < 2:  # If only one result is available
        return "single", results.iloc[0]  # Return single mode with that result

    top_score = results.iloc[0]["score"]  # Get highest ranked score

    second_score = results.iloc[1]["score"]  # Get second highest score

    diff = top_score - second_score  # Calculate confidence gap between top 2 results

    if diff > threshold:  # If the gap is large enough (high confidence)
        return "single", results.iloc[0]  # Return only the best result

    else:
        return "multiple", results.head(3)  # Otherwise return multiple possible results (uncertain case)

# Intent Detection


In [11]:
def detect_intent(query):
    q = query.lower()  # Convert query to lowercase

    if "difference" in q or "vs" in q or "compare" in q:
        return "compare"  # Comparison intent
    elif "why" in q:
        return "reason"  # Explanation intent
    elif "multiple" in q:
        return "multi"  # Multiple diseases intent
    else:
        return "normal"  # Default


# Comparison


In [12]:
def compare_diseases(results):
    if len(results) < 2:
        return "Not enough data to compare."  # Need at least 2 diseases

    d1, d2 = results.iloc[0], results.iloc[1]  # Get top 2 results

    return f"""
Comparison:

{d1['clean_name']} vs {d2['clean_name']}

Type:
- {d1['type']} vs {d2['type']}

Symptoms:
- {d1['Disease Definition']}
- {d2['Disease Definition']}

Treatment:
- {d1['Treatment']}
- {d2['Treatment']}
"""


# Reasoning Layer


In [13]:
def reasoning_answer(query, best):
    q = query.lower()
    disease_type = str(best.get("type", "")).lower()
    name = best.get("clean_name", "this disease")

    # 1. ANTIBIOTIC MISUSE (High Priority)
    # Checks if user asks for antibiotics for non-bacterial diseases
    if any(word in q for word in ["antibiotic", "antibacterial"]):
        if disease_type in ["fungal", "oomycete", "virus", "pest"]:
            return f"""
Warning:
Antibiotics do NOT work on {disease_type} diseases like {name}.

Recommended:
- Use appropriate fungicides or antivirals.
- Follow a proper spraying schedule.
"""

    # 2. FUNGICIDE / SPRAY MISUSE
    # Triggered if user suggests "fungicide" or generic "spray" for Viruses or Bacteria
    if any(word in q for word in ["fungicide", "spray", "chemical"]):
        
        # Case: Cassava Mosaic (Virus) + Fungicide
        if disease_type == "virus":
            return f"""
Warning:
Fungicides do NOT work on viral diseases like {name}.

Recommended:
- Remove and burn infected plants immediately.
- Control insect vectors (like whiteflies) that spread the virus.
"""
        
        # Case: Citrus Greening (Bacterial) + Generic Spray
        elif disease_type == "bacterial":
            return f"""
Warning:
Standard fungicides/sprays do NOT work on bacterial diseases like {name}.

Use instead:
- Copper-based bactericides.
- Integrated Pest Management (IPM) to control psyllids.
"""

    # 3. GENERAL VIRAL INSIGHT (Fallback)
    if disease_type == "virus":
        return f"""
Diagnosis Insight:
{name} is a viral disease.
No chemical cure (fungicide/antibiotic) is available.

Actions:
- Use virus-free planting material.
- Remove infected plants immediately.
"""

    return None

# Multi Disease


In [14]:
def multi_disease_answer(results):
    response = "Possible multiple diseases:\n\n"

    for _, row in results.iterrows():
        response += f"{row['clean_name']} ({row['type']})\n"  # List possible diseases

    response += "\nPlease give more details for accurate diagnosis."

    return response


# Final Answer


In [15]:
def answer_question(query):
    intent = detect_intent(query)  # Detect what the user wants (compare, reason, normal, etc.)

    mode, data = retrieve_with_confidence(query)  # Retrieve results with confidence check (single or multiple)

# Reasoning layer (high priority for single confident result)
    if mode == "single":
        best = data
        reasoning = reasoning_answer(query, best)  # Run reasoning rules on the best result only
        if reasoning:
            return reasoning  # If reasoning gives an answer, return immediately

# Compare intent
    if intent == "compare":
        results = hybrid_search(query)  # Get candidates using hybrid search

        results = rerank_full(query, results)  # Improve ranking using CrossEncoder

        return compare_diseases(results)  # Return comparison between top diseases

# Multiple possible diseases (uncertain case)
    if mode == "multiple":
        return multi_disease_answer(data)  # Return list of possible diseases

# Normal answer (most confident case)
    best = data  # Take best predicted result

    return f"""
Disease: {best['clean_name']} ({best['type']})

Definition:
{best['Disease Definition']}

Treatment:
{best['Treatment']}

Active Ingredient:
{best['Active Ingredient']}

Dosage:
{best['Dosage']}

Steps:
{best['Treatment Steps']}

Duration:
{best['Duration of Treatment']}

Recheck:
{best['Medical Consultation']}
"""
    

# saved model


In [16]:
save_path = r"D:\Fall 2026 (4)\CS 1st Semester\CS492 Project\Chat Bot"  # Define folder name where model files will be saved
os.makedirs(save_path, exist_ok=True)  # Create folder if it does not exist

# 1. Save DataFrame
df.to_pickle(os.path.join(save_path, "df.pkl"))  # Save the pandas DataFrame as a binary file (faster than CSV)

# 2. Save BM25 Model
with open(os.path.join(save_path, "bm25.pkl"), "wb") as f:  # Open file in write-binary mode
    pickle.dump(bm25, f)  # Save BM25 object into file

# 3. Save FAISS Index
faiss.write_index(index, os.path.join(save_path, "faiss.index"))  # Save FAISS vector index to disk (for fast similarity search later)

# 4. Save Embeddings
np.save(os.path.join(save_path, "embeddings.npy"), embeddings)  # Save embeddings as NumPy array (optional but useful for debugging or reuse)

# 5. Save Text Data
with open(os.path.join(save_path, "texts.pkl"), "wb") as f:  # Open file in write-binary mode
    pickle.dump(texts, f)  # Save original text data

# 6. Save Config 
config = {
    "embedding_model": "BAAI/bge-base-en-v1.5",
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2"
}

with open(os.path.join(save_path, "config.pkl"), "wb") as f:
    pickle.dump(config, f)

# Done
print("Model saved successfully!")  # Print confirmation message


Model saved successfully!


# Run Test


In [17]:
test_questions = [
    "What is tomato early blight?",
    "Tomato early blight vs late blight",
    "My plant has yellow leaves",
    "Can I use antibiotics for fungal disease?",
    "Can I use fungicide for bacterial disease?",
    "asdasdasd",
    "How to treat tomato mosaic virus?",
    "treatment for cassava mosaic using fungicide",
    "I don’t know what is happening",
    "My tomato has yellow leaves and spots"
]

for q in test_questions:
    print("Q:", q)
    print(answer_question(q))
    print("="*50)

Q: What is tomato early blight?

Disease: Tomato early_blight   (fungal)

Definition:
Fungal disease caused by Alternaria solani — dark concentric leaf spots, stem lesions, fruit rot.

Treatment:
Fungicides (e.g., chlorothalonil, mancozeb), crop rotation, pruning for airflow, resistant varieties.

Active Ingredient:
Chlorothalonil, Mancozeb

Dosage:
As per label instructions

Steps:
1) Remove infected leaves
2) Apply fungicide at early signs
3) Maintain proper spacing and airflow
4) Rotate crops and prune for light penetration

Duration:
Seasonal, repeat under high humidity

Recheck:
Recheck after 7–10 days to assess blight reduction

Q: Tomato early blight vs late blight

Comparison:

Tomato late_blight   vs Tomato early_blight  

Type:
- fungal vs fungal

Symptoms:
- Caused by Phytophthora infestans — water-soaked spots on leaves, gray-white growth underside, fruit rot.
- Fungal disease caused by Alternaria solani — dark concentric leaf spots, stem lesions, fruit rot.

Treatment:
- R

In [ ]:
while True:
    q = input("\nAsk: ")
    print(answer_question(q))
    